# Classifying Sounds From UrbanSound8K

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import soundata
import librosa
import os


## Load Dataset

In [ ]:
# Load .env file
load_dotenv()

# Define home path for soundata
data_home = os.getenv("SOUND_DATA_HOME")

# Load metadata from csv file
metadata_path = data_home + "/urbansound8k/metadata/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Only examine samples from these classes
selected_classes = ['dog_bark', 'car_horn', 'children_playing', 'drilling']

# Filter metadata to only contain selected classes
metadata = metadata[metadata['class'].isin(selected_classes)]

# Load dataset
dataset = soundata.initialize("urbansound8k", data_home=data_home)

# Filter clips based on selected classes
subset = [clip for clip in dataset.clip_ids if dataset.clip(clip).tags.labels[0] in selected_classes]

## Function for Extracting Features Using Librosa

In [ ]:
def extract_features(clip):

    # Number of Mel-Frequency Cepstral Coefficients to extract
    n_mfcc = 13

    wave, sample_rate = clip.audio

    # Extract MFCC
    mfcc = librosa.feature.mfcc(y=wave, sr=sample_rate, n_mfcc=n_mfcc)

    # Extract mean and std for mfcc and first and second delta of mfcc
    # Examining deltas allows for insights into how mfcc (and therefore tone) changes over time

    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.concatenate([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
        np.mean(delta, axis=1), np.std(delta, axis=1),
        np.mean(delta2, axis=1), np.std(delta2, axis=1)
    ])

    return features
